# BloomWiSARD — Sweep de Anomalia em IoT (TON_IoT)

Sweep completo do modelo **BloomWiSARD** sobre as 7 bases de telemetria do
**TON_IoT** (UNSW Canberra Cyber). Cobre as duas tarefas definidas em `PLANO.md`:

- **Tarefa binária**: `normal` vs `ataque`
- **Tarefa multi-classe**: classificação por tipo de ataque

Hiperparâmetros varridos (PLANO §3.1 + §3.2 BloomWiSARD):

| Eixo | Valores |
|---|---|
| Tipo de termômetro | Simple, Distributive, Gaussian, Exponential |
| Tamanho do termômetro | {2, 4, 8, 16, 32, 64} |
| Tamanho do endereço | {4, 8, 12, 16, 20, 24, 28, 32} |
| `hashMode` | murmur, simhash, h3 |
| `numHashes` | {2, 4, 8} |
| `filterSize` (numBits) | {16, 64, 256, 1024} |

Resultados em `results/bloomwisard/<base>__<task>.jsonl`.


## 0.1 Setup do ambiente

Instala dependências diretamente no notebook — sem venv externo.

In [13]:
# Dependências gerenciadas pelo uv (pyproject.toml).
# Se rodar fora do projeto (ex: Colab), descomente:
# import sys
# !{sys.executable} -m ensurepip --upgrade
# !{sys.executable} -m pip install git+https://github.com/muanlartins/wisardpkg.git@muanlartins
# !{sys.executable} -m pip install "pandas>=2.0" scikit-learn scipy numpy matplotlib tqdm

## 0.2 Imports e constantes

In [14]:
import os, json, time, pickle, math, warnings, shutil
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, balanced_accuracy_score,
                             roc_auc_score, confusion_matrix)
from tqdm.auto import tqdm
import wisardpkg as wp

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

DATA_DIR    = Path('../../data/toniot/')
RESULTS_DIR = Path('./results/')

EXPECTED = ['Fridge', 'Garage_Door', 'GPS_Tracker', 'Modbus',
            'Motion_Light', 'Thermostat', 'Weather']

# True → roda apenas a primeira combinação válida de hiperparâmetros por base/tarefa
DRY_RUN = False

print(f'wisardpkg {wp.__version__}')

wisardpkg 2.0.0a7


## 1. Carregar e limpar os 7 CSVs

A função `clean_df` é obrigatória para corrigir espaços em branco e
inconsistências nos CSVs crus do TON_IoT (ver `01_eda_telemetry.ipynb §1`).


In [15]:
def clean_df(df):
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip().str.lower()
    if 'sphone_signal' in df.columns:
        df['sphone_signal'] = (df['sphone_signal']
                               .map({'0': 0, '1': 1, 'false': 0, 'true': 1})
                               .astype('Int64'))
    if 'label' in df.columns:
        df['label'] = pd.to_numeric(df['label'], errors='coerce').astype('Int64')
    return df


def find_csv(device):
    if not DATA_DIR.exists():
        return None
    target = f'train_test_iot_{device}.csv'.lower()
    for p in DATA_DIR.iterdir():
        if p.name.lower() == target:
            return p
    return None


dfs = {}
missing = []
for device in EXPECTED:
    p = find_csv(device)
    if p is None:
        missing.append(device)
        continue
    dfs[device] = clean_df(pd.read_csv(p))

print(f'Carregadas {len(dfs)} / {len(EXPECTED)} bases.')
for device, df in dfs.items():
    print(f'  {device:<13}  {df.shape[0]:>6,} x {df.shape[1]}')
if missing:
    print(f'\nFALTANDO: {missing}')
    print(f'Esperado em: {DATA_DIR.resolve()}')
DATA_AVAILABLE = len(dfs) == len(EXPECTED)
assert DATA_AVAILABLE, 'Faltam CSVs — ver README.md'

Carregadas 7 / 7 bases.
  Fridge         39,944 x 6
  Garage_Door    39,587 x 6
  GPS_Tracker    38,960 x 6
  Modbus         31,106 x 8
  Motion_Light   39,488 x 6
  Thermostat     32,774 x 6
  Weather        39,260 x 7


## 2. Identificação de features

In [16]:
META_COLS = {'date', 'time', 'label', 'type'}


def feature_cols(df):
    num, cat = [], []
    for c in df.columns:
        if c in META_COLS:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            num.append(c)
        else:
            cat.append(c)
    return num, cat


def cat_encoded_bits(unique_values):
    # 2 valores -> 1 bit (binaria); k>2 -> k bits (one-hot)
    return 1 if len(unique_values) == 2 else len(unique_values)


for device, df in dfs.items():
    num, cat = feature_cols(df)
    card = sum(cat_encoded_bits(df[c].unique()) for c in cat)
    print(f'{device:<13}  num={num}  cat={cat}  cat_bits={card}')

Fridge         num=['fridge_temperature']  cat=['temp_condition']  cat_bits=1
Garage_Door    num=['sphone_signal']  cat=['door_state']  cat_bits=1
GPS_Tracker    num=['latitude', 'longitude']  cat=[]  cat_bits=0
Modbus         num=['fc1_read_input_register', 'fc2_read_discrete_value', 'fc3_read_holding_register', 'fc4_read_coil']  cat=[]  cat_bits=0
Motion_Light   num=['motion_status']  cat=['light_status']  cat_bits=1
Thermostat     num=['current_temperature', 'thermostat_status']  cat=[]  cat_bits=0
Weather        num=['temperature', 'pressure', 'humidity']  cat=[]  cat_bits=0


## 3. Codificação binária (termômetros + categóricas)

O termômetro é ajustado **só no treino** para evitar leakage do teste.
Política de bits categóricos: binária → 1 bit; k-ária → k bits (one-hot).


In [17]:
THERM_BUILDERS = {
    'Simple':       lambda size, mins, maxs: wp.SimpleThermometer(size, minimum=min(mins), maximum=max(maxs)),
    'Distributive': lambda size, mins, maxs: wp.DistributiveThermometer(size),
    'Gaussian':     lambda size, mins, maxs: wp.GaussianThermometer(size),
    'Exponential':  lambda size, mins, maxs: wp.ExponentialThermometer(size),
}


def fit_encoder(df_train, num_cols, therm_type, size):
    if not num_cols:
        return None
    vals = df_train[num_cols].astype(float).values
    mins = vals.min(axis=0).tolist()
    maxs = vals.max(axis=0).tolist()
    therm = THERM_BUILDERS[therm_type](size, mins, maxs)
    if therm_type != 'Simple':
        therm.fit(vals.tolist())
    return therm


def cat_categories(df_train, cat_cols):
    return {c: sorted(df_train[c].unique()) for c in cat_cols}


def encode_dataset(df, num_cols, cat_cols, cat_values, therm):
    if num_cols:
        num_vals = df[num_cols].astype(float).values
        bit_rows = [therm.transform(row.tolist()).list() for row in num_vals]
    else:
        bit_rows = [[] for _ in range(len(df))]
    if cat_cols:
        cat_vals_arr = df[cat_cols].values
        for i, row in enumerate(cat_vals_arr):
            for c_idx, c in enumerate(cat_cols):
                v = row[c_idx]
                cats = cat_values[c]
                if len(cats) == 2:
                    bit_rows[i].append(1 if v == cats[1] else 0)
                else:
                    bit_rows[i].extend([1 if v == cat else 0 for cat in cats])
    return bit_rows


def encoded_input_bits(n_num, cat_values, therm_size):
    cat_bits = sum(cat_encoded_bits(v) for v in cat_values.values())
    return n_num * therm_size + cat_bits


def make_mapping(input_bits, address_size):
    """Mapeamento aleatório determinístico (seed=0) conforme PLANO §3.2."""
    rng = np.random.RandomState(0)
    n_rams = math.ceil(input_bits / address_size)
    indexes = (rng.permutation(n_rams * address_size) % input_bits).tolist()
    return wp.RandomMapping(indexes, address_size)

## 4. Split estratificado

`random_state=0`, `test_size=0.3`, estratificado por `type`.
Mesmo split para as duas tarefas — a diferença é apenas a coluna alvo (`label` vs `type`).


In [18]:
def make_split(df, task, test_size=0.3, random_state=0):
    if task == 'binary':
        y = df['label'].astype(int).map({0: 'normal', 1: 'attack'}).values
    elif task == 'multiclass':
        y = df['type'].values
    else:
        raise ValueError(task)
    idx_train, idx_test = train_test_split(
        np.arange(len(df)),
        test_size=test_size,
        random_state=random_state,
        stratify=df['type'].values,
    )
    return (df.iloc[idx_train].reset_index(drop=True),
            df.iloc[idx_test].reset_index(drop=True),
            y[idx_train], y[idx_test])

## 5. Métricas (schema PLANO §4)

In [19]:
def compute_metrics(y_true, y_pred, y_score=None, task='binary',
                    score_classes=None):
    out = {
        'accuracy':           float(accuracy_score(y_true, y_pred)),
        'precision_macro':    float(precision_score(y_true, y_pred, average='macro',    zero_division=0)),
        'precision_weighted': float(precision_score(y_true, y_pred, average='weighted', zero_division=0)),
        'recall_macro':       float(recall_score(y_true, y_pred, average='macro',    zero_division=0)),
        'recall_weighted':    float(recall_score(y_true, y_pred, average='weighted', zero_division=0)),
        'f1_macro':           float(f1_score(y_true, y_pred, average='macro',    zero_division=0)),
        'f1_weighted':        float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'balanced_accuracy':  float(balanced_accuracy_score(y_true, y_pred)),
        'auc_roc':            None,
    }
    if y_score is not None:
        try:
            score_arr = np.asarray(y_score, dtype=float)
            if task == 'binary':
                if score_arr.ndim == 1:
                    pos_score = score_arr
                else:
                    pos_idx = score_classes.index('attack')
                    pos_score = score_arr[:, pos_idx]
                out['auc_roc'] = float(roc_auc_score(y_true == 'attack', pos_score))
            else:
                out['auc_roc'] = float(roc_auc_score(
                    y_true, score_arr, multi_class='ovr', average='macro',
                    labels=score_classes,
                ))
        except Exception:
            pass
    labels = sorted(set(list(y_true) + list(y_pred)))
    cm = confusion_matrix(y_true, y_pred, labels=labels).tolist()
    return out, cm, labels


def ranks_to_score_matrix(rank_list, classes):
    M = np.zeros((len(rank_list), len(classes)), dtype=float)
    for i, d in enumerate(rank_list):
        total = sum(d.values()) or 1
        for k, cls in enumerate(classes):
            M[i, k] = d.get(cls, 0) / total
    return M

## 6. Memória e latência — BloomWiSARD

**Memória teórica**: `n_classes × n_rams × ceil(filterSize / 8)` bytes,
onde `filterSize` = numBits do Bloom filter e `n_rams = ceil(input_bits / addressSize)`.

**Serializada**: estimada via `pickle` (BloomWiSARD não expõe `.json()`).


In [20]:
def measure_memory_bloom(clf, input_bits, address_size, n_classes, filter_size):
    n_rams = math.ceil(input_bits / address_size)
    bytes_per_filter = math.ceil(filter_size / 8)
    theoretical = n_classes * n_rams * bytes_per_filter
    try:
        serialized = len(pickle.dumps(clf))
    except Exception:
        serialized = theoretical
    return serialized, theoretical


def measure_latency(clf, X_test_lists, n_iter=1000):
    if not X_test_lists:
        return None
    n = len(X_test_lists)
    pool_size = min(n, n_iter)
    pool = [wp.DataSet([X_test_lists[i]]) for i in range(pool_size)]
    for ds in pool[:min(20, pool_size)]:
        _ = clf.classify(ds)
    times = []
    for i in range(n_iter):
        ds = pool[i % pool_size]
        t0 = time.perf_counter_ns()
        _ = clf.classify(ds)
        times.append((time.perf_counter_ns() - t0) / 1000.0)
    times.sort()
    return float(times[len(times) // 2])

## 7. Grid de hiperparâmetros — BloomWiSARD

Produto cartesiano dos 6 eixos:
- Eixos comuns: termômetro (4) × thermoSize (6) × addressSize (8) = 192
- BloomWiSARD: hashMode (3) × numHashes (3) × filterSize (4) = 36
- **Total: 6.912 configs** por base por tarefa (menos skips de `addressSize > input_bits`)


In [21]:
THERMO_SIZES  = [2, 4, 8, 16, 32, 64]
ADDRESS_SIZES = [4, 8, 12, 16, 20, 24, 28, 32]
THERMOMETERS  = ['Simple', 'Distributive', 'Gaussian', 'Exponential']

HASH_MODES    = ['murmur', 'simhash', 'h3']
NUM_HASHES    = [2, 4, 8]
FILTER_SIZES  = [16, 64, 256, 1024]  # numBits no construtor


def iter_grid_bloom(n_num, cat_card,
                    thermo_sizes=THERMO_SIZES,
                    address_sizes=ADDRESS_SIZES,
                    thermometers=THERMOMETERS,
                    hash_modes=HASH_MODES,
                    num_hashes_list=NUM_HASHES,
                    filter_sizes=FILTER_SIZES):
    for th in thermometers:
        for ts in thermo_sizes:
            input_bits = n_num * ts + cat_card
            for ad in address_sizes:
                skipped = ad > input_bits
                for hm in hash_modes:
                    for nh in num_hashes_list:
                        for fs in filter_sizes:
                            yield {
                                'thermometer':  th,
                                'thermo_size':  ts,
                                'address_size': ad,
                                'input_bits':   input_bits,
                                'hash_mode':    hm,
                                'num_hashes':   nh,
                                'filter_size':  fs,
                                'skipped':      skipped,
                            }


print('Configs por base:')
for device, df in dfs.items():
    num, cat = feature_cols(df)
    card = sum(cat_encoded_bits(df[c].unique()) for c in cat)
    total = sum(1 for _ in iter_grid_bloom(len(num), card))
    valid = sum(1 for c in iter_grid_bloom(len(num), card) if not c['skipped'])
    print(f'  {device:<13}  total={total:>6,}  validas={valid:>6,}  skip={total-valid:>5,}')

Configs por base:
  Fridge         total= 6,912  validas= 3,312  skip=3,600
  Garage_Door    total= 6,912  validas= 3,312  skip=3,600
  GPS_Tracker    total= 6,912  validas= 4,464  skip=2,448
  Modbus         total= 6,912  validas= 5,472  skip=1,440
  Motion_Light   total= 6,912  validas= 3,312  skip=3,600
  Thermostat     total= 6,912  validas= 4,464  skip=2,448
  Weather        total= 6,912  validas= 4,896  skip=2,016


## 8. Schema de resultado (PLANO §4)

In [22]:
def result_dict(model_name, base, task, config, metrics, cm, labels,
                input_info, model_hyperparams=None, machine=None,
                skipped=False, skipped_reason=None):
    return {
        'model':             model_name,
        'base':              base,
        'task':              task,
        'encoder':           {'type': config['thermometer'], 'size': config['thermo_size']},
        'addressSize':       config['address_size'],
        'model_hyperparams': model_hyperparams or {},
        'split':             {'random_state': 0, 'test_size': 0.3, 'stratified': True},
        'input':             input_info,
        'skipped':           skipped,
        'skipped_reason':    skipped_reason,
        'metrics':           metrics,
        'confusion_matrix':  cm,
        'labels':            labels,
        'machine':           machine,
        'wisardpkg_version': wp.__version__,
        'timestamp':         datetime.now(timezone.utc).isoformat(timespec='seconds'),
    }


def save_result(result, model_name, base, task):
    out_dir = RESULTS_DIR / model_name.lower()
    out_dir.mkdir(parents=True, exist_ok=True)
    fname = f'{base}__{task}.jsonl'
    with open(out_dir / fname, 'a') as f:
        f.write(json.dumps(result) + '\n')
    return out_dir / fname

## 9. Sweep completo — BloomWiSARD

Roda todas as bases × tarefas × configs do grid. Cada config gera uma linha
no `.jsonl` correspondente. Configs inválidas (`addressSize > input_bits`)
são registradas com `skipped=true` sem treinar modelo.

> **Re-rodar limpa** os JSONLs anteriores para evitar duplicatas.


In [23]:
MACHINE = 'local'   # ajuste: 'colab-cpu', 'local', etc.
TASKS   = ['binary', 'multiclass']

# Limpa resultados anteriores deste sweep (re-roda do zero)
bloom_results_dir = RESULTS_DIR / 'bloomwisard'
if bloom_results_dir.exists():
    shutil.rmtree(bloom_results_dir)
bloom_results_dir.mkdir(parents=True, exist_ok=True)

EMPTY_METRICS = {k: None for k in [
    'accuracy', 'precision_macro', 'precision_weighted',
    'recall_macro', 'recall_weighted', 'f1_macro', 'f1_weighted',
    'balanced_accuracy', 'auc_roc',
    'memory_bytes_serialized', 'memory_bytes_theoretical',
    'train_time_s', 'inference_latency_us',
]}

for device in EXPECTED:
    df = dfs[device]
    num, cat = feature_cols(df)
    cat_card = sum(cat_encoded_bits(df[c].unique()) for c in cat)

    for task in TASKS:
        df_train, df_test, y_train, y_test = make_split(df, task)
        cat_values = cat_categories(df_train, cat)
        cat_bits   = sum(cat_encoded_bits(v) for v in cat_values.values())
        classes    = sorted(set(y_train))

        grid = list(iter_grid_bloom(len(num), cat_card))
        if DRY_RUN:
            grid = [c for c in grid if not c['skipped']][:1]
        pbar = tqdm(grid, desc=f'{device}/{task}', leave=False)

        for config in pbar:
            input_info = {
                'n_features_num': len(num),
                'n_features_cat': len(cat),
                'cat_bits':       cat_bits,
                'bits_total':     config['input_bits'],
            }
            hp = {
                'hashMode':     config['hash_mode'],
                'numHashes':    config['num_hashes'],
                'filterSize':   config['filter_size'],
                'bleaching':    'BestBleaching',
                'mapping_seed': 0,
            }

            if config['skipped']:
                res = result_dict(
                    'BloomWiSARD', device, task, config,
                    metrics=EMPTY_METRICS, cm=[], labels=[],
                    input_info=input_info, model_hyperparams=hp,
                    machine=MACHINE, skipped=True,
                    skipped_reason='addressSize > input_bits',
                )
                save_result(res, 'BloomWiSARD', device, task)
                continue

            therm   = fit_encoder(df_train, num, config['thermometer'], config['thermo_size'])
            mapping = make_mapping(config['input_bits'], config['address_size'])

            X_train = encode_dataset(df_train, num, cat, cat_values, therm)
            X_test  = encode_dataset(df_test,  num, cat, cat_values, therm)

            ds_train = wp.DataSet(X_train, list(y_train))
            ds_test  = wp.DataSet(X_test)

            clf = wp.BloomWisard(
                config['address_size'],
                config['filter_size'],
                config['num_hashes'],
                hashMode=config['hash_mode'],
                mapping=mapping,
                classificationMethod=wp.BestBleaching(),
            )
            t0 = time.perf_counter()
            clf.train(ds_train)
            train_time_s = time.perf_counter() - t0

            y_pred   = np.array(clf.classify(ds_test))
            rank_out = clf.rank(ds_test)
            y_score  = ranks_to_score_matrix(rank_out, classes)

            metrics, cm, labels = compute_metrics(
                y_test, y_pred, y_score=y_score, task=task, score_classes=classes,
            )
            ser, theo = measure_memory_bloom(
                clf, config['input_bits'], config['address_size'],
                len(classes), config['filter_size'],
            )
            lat = measure_latency(clf, X_test[:200], n_iter=500)
            metrics.update({
                'memory_bytes_serialized':  int(ser),
                'memory_bytes_theoretical': int(theo),
                'train_time_s':             round(train_time_s, 4),
                'inference_latency_us':     round(lat, 2) if lat is not None else None,
            })

            res = result_dict(
                'BloomWiSARD', device, task, config,
                metrics=metrics, cm=cm, labels=labels,
                input_info=input_info, model_hyperparams=hp,
                machine=MACHINE,
            )
            save_result(res, 'BloomWiSARD', device, task)

        pbar.close()
        jsonl = bloom_results_dir / f'{device}__{task}.jsonl'
        if jsonl.exists():
            lines = jsonl.read_text().strip().splitlines()
            done  = sum(1 for l in lines if not json.loads(l).get('skipped'))
            skip  = len(lines) - done
            print(f'{device:<13} {task:<11}  {done:>5} rodados  {skip:>5} skipped')

print('\nSweep concluido.')

Fridge        binary        3312 rodados   3600 skipped


Fridge        multiclass    3312 rodados   3600 skipped


Garage_Door   binary        3312 rodados   3600 skipped


Garage_Door   multiclass    3312 rodados   3600 skipped


GPS_Tracker   binary        4464 rodados   2448 skipped


...

Motion_Light  multiclass    3312 rodados   3600 skipped


Thermostat    binary        4464 rodados   2448 skipped


Thermostat    multiclass    4464 rodados   2448 skipped


Weather       binary        4896 rodados   2016 skipped


Weather       multiclass    4896 rodados   2016 skipped

Sweep concluido.


## 10. Análise rápida dos resultados

Top-5 configs por base e tarefa ordenadas por F1 macro (excluindo configs puladas).
Para a análise completa com fronteira de Pareto e matrizes de confusão,
ver `03_analise_final.ipynb`.


In [24]:
def load_results(model_name='bloomwisard'):
    rows = []
    res_dir = RESULTS_DIR / model_name
    if not res_dir.exists():
        return pd.DataFrame()
    for fpath in sorted(res_dir.glob('*.jsonl')):
        for line in fpath.read_text().strip().splitlines():
            rows.append(json.loads(line))
    return pd.DataFrame(rows)


res = load_results()
if res.empty:
    print('Nenhum resultado ainda — rode a celula do sweep primeiro.')
else:
    metrics_df = pd.json_normalize(res['metrics'])
    bloom_df = pd.concat([
        res[['model', 'base', 'task', 'addressSize']].reset_index(drop=True),
        pd.json_normalize(res['encoder']).add_prefix('enc_'),
        pd.json_normalize(res['model_hyperparams']).add_prefix('hp_'),
        metrics_df[['accuracy', 'f1_macro', 'f1_weighted',
                    'memory_bytes_theoretical', 'train_time_s']],
        res['skipped'].reset_index(drop=True),
    ], axis=1)

    print('=== Top-5 por base x tarefa (F1 macro, sem skips) ===\n')
    for (base, task), grp in bloom_df[~bloom_df['skipped']].groupby(['base', 'task']):
        top = (grp.nlargest(5, 'f1_macro')
               [['enc_type', 'enc_size', 'addressSize', 'hp_hashMode',
                 'hp_numHashes', 'hp_filterSize', 'accuracy', 'f1_macro',
                 'memory_bytes_theoretical', 'train_time_s']]
               .round(4))
        print(f'--- {base} / {task} ---')
        print(top.to_string(index=False))
        print()

=== Top-5 por base x tarefa (F1 macro, sem skips) ===

--- Fridge / binary ---
enc_type  enc_size  addressSize hp_hashMode  hp_numHashes  hp_filterSize  accuracy  f1_macro  memory_bytes_theoretical  train_time_s
  Simple         4            4      murmur             2             16    0.3755     0.273                       8.0        0.0031
  Simple         4            4      murmur             2             64    0.3755     0.273                      32.0        0.0036
  Simple         4            4      murmur             2            256    0.3755     0.273                     128.0        0.0032
  Simple         4            4      murmur             2           1024    0.3755     0.273                     512.0        0.0031
  Simple         4            4      murmur             4             16    0.3755     0.273                       8.0        0.0035

--- Fridge / multiclass ---
enc_type  enc_size  addressSize hp_hashMode  hp_numHashes  hp_filterSize  accuracy  f1_macro  